In [18]:
import core.dataset
import core.experiment
import core.grid
import numpy as np
from scripts.analyze_mlflow_runs import (
    analyze_unified_mlruns,
    aggregate_metrics,
    print_summary,
    plot_metrics,
)
from helpers import utils
from scripts import diagnose_fold_differences as fold_dx
import re
import os
from collections import defaultdict
from reload_recursive import reload_recursive
from pathlib import Path
from IPython.display import clear_output
from tqdm.notebook import tqdm
import pandas as pd

In [19]:
reload_recursive(core.dataset)
from core.dataset import Dataset

reload_recursive(core.experiment)
from core.experiment import Experiment

reload_recursive(core.grid)
from core.grid import ExperimentGrid

reload_recursive(core.configs)
from core.configs import AlgoConfig

reload_recursive(utils)

import scripts.compile_run_metrics
reload_recursive(scripts.compile_run_metrics)


In [20]:
import sys
from loguru import logger

# Map module names (or substrings) to minimum log levels.
# Anything not listed gets the default level.
MODULE_LEVELS = {
    "generate_fold_predictions": "WARNING",  # quiet during predict()
    "inference": "INFO",
    "grid": "DEBUG",
}
DEFAULT_LEVEL = "INFO"


def _module_filter(record):
    name = record["name"]  # e.g. "scripts.generate_fold_predictions"
    for module, level in MODULE_LEVELS.items():
        if module in name:
            return record["level"].no >= logger.level(level).no
    return record["level"].no >= logger.level(DEFAULT_LEVEL).no


logger.remove()  # drop the default stderr handler
logger.add(
    sys.stderr,
    filter=_module_filter,
    format="{time:HH:mm:ss} | {level:<7} | {name} - {message}",
)

MODULE_LEVELS["generate_fold_predictions"] = "INFO"
logger.remove()
logger.add(sys.stderr, filter=_module_filter)

2

In [37]:
def get_experiment(dataset_name, experiment_name):
    dataset = Dataset(dataset_name)
    return Experiment.from_run_dir(dataset.work_home/experiment_name)

In [55]:
reload_recursive(core.experiment)
from core.experiment import Experiment
reload_recursive(core.grid)
from core.grid import ExperimentGrid
import scripts.compile_run_metrics
reload_recursive(scripts.compile_run_metrics)
from scripts.compile_run_metrics import compile_all_metrics0, compile_experiment_metrics

params_to_gather = [
    "learning_rate",
    "crop_ratios",
    "num_crops_per_image",
    "batch_size",
    "loss#weight",
    "loss#include_background",
]

dataset_name = "roi_train2"
dataset = Dataset(dataset_name)
stage = "stage4_sweep_dicece_wts"

dataset = Dataset(dataset_name)
experiment_home = dataset.work_home / stage
grid = ExperimentGrid.from_home_dir(experiment_home)

experiment_dict = {
    "roi_train1": ["roi_train1_segresnet"],
    "roi_train2": ["run1", "run2", "run3", "test_dicece_lambda/run1"],
    "roi_train2_t1": ["test_dicece_lambda/run1"]
}

grid_dict = {
    "roi_train2": [
        "stage4_sweep_dicece_wts",
        "stage5_sweep_dicecewt_nbatch",
        "stage3_numcrops_bkd_constwt115",
        "stage1_crop_lr_sweep",
        "stage2_numcrops_dicece",
        "test_dicece_lambda"
    ],
    "roi_train2_t1": ["sweep_dicecewts"]
}

experiments = []
for dataset_name, experiment_names in experiment_dict.items():
    dataset = Dataset(dataset_name)
    for name in experiment_names:
        experiments.append(Experiment.from_run_dir(dataset.work_home/name))
        
grids = []
for dataset_name, grid_names in grid_dict.items():
    dataset = Dataset(dataset_name)
    for name in grid_names:
        grids.append(ExperimentGrid.from_home_dir(dataset.work_home/name))

MODULE_LEVELS["compile_run_metrics"] = "DEBUG"
# data = compile_all_metrics(experiments=experiments,
#                     grids=grids, 
#                     params_to_gather=params_to_gather, 
#                     use_cache=True,)
data = compile_all_metrics0(experiments=experiments,
                    grids=grids, 
                    params_to_gather=params_to_gather, 
                    use_cache=False,)

# clear_output()

2026-03-26 03:08:22.645 | INFO     | scripts.compile_run_metrics:compile_experiment_metrics:354 - Starting Experiment roi_train1/roi_train1_segresnet
2026-03-26 03:08:23.247 | DEBUG    | scripts.compile_run_metrics:load_or_cache_run:215 - loading mlflow took (0.57s)
2026-03-26 03:08:25.748 | DEBUG    | scripts.compile_run_metrics:load_or_cache_run:226 - Building cases took (2.50s)
2026-03-26 03:08:25.751 | DEBUG    | scripts.compile_run_metrics:load_or_cache_run:252 - Cached: roi_train1_roi_train1_segresnet.pkl
2026-03-26 03:08:25.752 | INFO     | scripts.compile_run_metrics:compile_experiment_metrics:354 - Starting Experiment roi_train2/run1
2026-03-26 03:08:26.432 | DEBUG    | scripts.compile_run_metrics:load_or_cache_run:215 - loading mlflow took (0.62s)
2026-03-26 03:08:30.671 | DEBUG    | scripts.compile_run_metrics:load_or_cache_run:226 - Building cases took (4.24s)
2026-03-26 03:08:30.675 | DEBUG    | scripts.compile_run_metrics:load_or_cache_run:252 - Cached: roi_train2_run1.pk

KeyboardInterrupt: 

In [44]:
data

,learning_rate,include_background,batch_size,crop_ratios,ID,weight,num_crops_per_image,status,rim/acc_mean,rim/acc_std,...,fold3-lesion/acc_final,fold3-lesion/acc_max,fold4-lesion/acc_final,fold4-lesion/acc_max,train/loss_mean,train/loss_std,train/loss_min,val/acc_max,val/acc_mean,val/acc_std
0,0.0002,true,1,none,roi_train1/roi_train1_segresnet,none,1,complete,0.3232,0.1217,...,0.6454,0.6547,0.6520,0.6659,0.6411,0.0660,0.5697,0.5670,0.4603,0.0743
1,0.0004,true,1,none,roi_train2/run1,none,1,complete,0.1776,0.1039,...,0.7240,0.7279,0.7191,0.7193,0.7692,0.0566,0.6979,0.5528,0.4255,0.0700
2,0.0002,true,1,none,roi_train2/run2,none,1,complete,0.3022,0.1289,...,0.6802,0.6866,0.6532,0.6649,0.6366,0.0589,0.5744,0.6196,0.4703,0.0816
3,0.0002,true,1,none,roi_train2/run3,none,1,complete,0.3149,0.1342,...,0.6955,0.7030,0.6655,0.6785,0.6088,0.0535,0.5574,0.6498,0.4843,0.0817
4,0.0002,true,1,none,roi_train2/test_dicece_lambda/run1,"1.0,1.0,5.0",4,complete,0.3927,0.1208,...,0.6806,0.6869,0.6658,0.6707,1.2934,0.0460,1.2532,0.6612,0.5171,0.0739
5,0.0002,true,1,none,roi_train2_t1/test_dicece_lambda/run1,"1.0,1.0,5.0",4,complete,0.3896,0.1184,...,0.6807,0.6878,0.6825,0.6925,1.2879,0.0433,1.2503,0.6636,0.5257,0.0692
6,0.0002,true,1,none,roi_train2/stage4_sweep_dicece_wts/run7,"1.0,2.0,5.0",1,complete,0.3920,0.1128,...,0.6867,0.6938,0.6622,0.6682,2.6781,0.1286,2.5621,0.6527,0.5190,0.0733
7,0.0002,true,1,none,roi_train2/stage4_sweep_dicece_wts/run8,"1.0,3.0,5.0",4,complete,0.3626,0.1190,...,0.6844,0.6936,0.6720,0.6725,2.7910,0.1791,2.6287,0.6500,0.5060,0.0760
8,0.0002,true,1,none,roi_train2/stage4_sweep_dicece_wts/run9,"1.0,5.0,5.0",1,complete,0.3341,0.1152,...,0.6915,0.6999,0.6595,0.6601,3.0211,0.2814,2.7657,0.6243,0.4898,0.0751
9,0.0002,false,2,none,roi_train2/stage5_sweep_dicecewt_nbatch/run1,"1.0,1.0,5.0",4,complete,0.4251,0.1067,...,0.6696,0.6806,0.6545,0.6614,3.8286,0.1078,3.7311,0.6552,0.5307,0.0683


In [39]:
exp1 = get_experiment("roi_train1", "roi_train1_segresnet")
exp2 = get_experiment("roi_train2", "run2")


In [ ]:
from helpers.utils import dice_score

# dataset_name = "roi_train2"
# experiment
# experiment = 
cases = pd.DataFrame(exp2.cases).set_index(["subid", "lesion_index"])
dice_scores = []
for case_index in [(1131,1), (2041,5), (1348,6), (2131,1), (1215, 3)]:
    label = cases.loc[case_index, "label"]
    inference = cases.loc[case_index, "inference"]
    dice = dice_score(label, inference, seg1_val=2, seg2_val=2)
    dice_scores.append([*case_index, cases.loc[case_index, "split"], dice])
dice_scores

[[1131, 1, 'fold1', np.float64(0.738562091503268)],
 [2041, 5, 'fold2', np.float64(0.2807017543859649)],
 [1348, 6, 'fold3', np.float64(0.6974358974358974)],
 [2131, 1, 'testing', np.float64(0.6951048951048951)],
 [1215, 3, 'fold1', np.float64(0.550561797752809)]]

In [50]:

dice_scores = defaultdict(list)
for (subid, lesion_index), item in cases.iterrows():
    if item['case_type'] == "PRL":
        dice_scores[item['split']].append({"subid": subid, "lesion_index": lesion_index,
                                     "dice": dice_score(item["label"], item["inference"], 2, 2)})
        

In [52]:
dice_scores['testing']

[{'subid': 1293, 'lesion_index': 1, 'dice': np.float64(0.27232796486090777)},
 {'subid': 1074, 'lesion_index': 3, 'dice': np.float64(0.44676409185803756)},
 {'subid': 1080, 'lesion_index': 3, 'dice': np.float64(0.502970297029703)},
 {'subid': 2131, 'lesion_index': 1, 'dice': np.float64(0.6951048951048951)},
 {'subid': 1011, 'lesion_index': 7, 'dice': np.float64(0.5601436265709157)},
 {'subid': 1529, 'lesion_index': 5, 'dice': np.float64(0.562268803945746)},
 {'subid': 2041, 'lesion_index': 1, 'dice': np.float64(0.48138437336130047)},
 {'subid': 1348, 'lesion_index': 1, 'dice': np.float64(0.555)},
 {'subid': 1529, 'lesion_index': 11, 'dice': np.float64(0.35526315789473684)}]

In [ ]:
ds = Dataset("roi_train1")
exp = Experiment.from_run_dir("/media/smbshare/srs-9/prl_project/training/roi_train1_segresnet", ds)

In [40]:
import scripts.compile_run_metrics
reload_recursive(scripts.compile_run_metrics)
from scripts.compile_run_metrics import compile_all_metrics

dataset_name = "roi_train2"
stages = [
    "stage4_sweep_dicece_wts",
    "stage5_sweep_dicecewt_nbatch",
    "stage3_numcrops_bkd_constwt115",
    "stage1_crop_lr_sweep",
    "stage2_numcrops_dicece",
    "test_dicece_lambda"
]
params_to_gather = [
    "learning_rate",
    "crop_ratios",
    "num_crops_per_image",
    "batch_size",
    "loss#weight",
    "loss#include_background",
]
extra_runs = [
    ("roi_train1", "/media/smbshare/srs-9/prl_project/training/roi_train1_segresnet"),
    ("standalone_run1", "/media/smbshare/srs-9/prl_project/training/roi_train2/run1"),
    ("standalone_run2", "/media/smbshare/srs-9/prl_project/training/roi_train2/run2"),
    ("standalone_run3", "/media/smbshare/srs-9/prl_project/training/roi_train2/run3"),
]

skip = {
    "stage2_numcrops_dicece": [f"run{i}" for i in [5,6,7,8,9,10,11,12]]
}

MODULE_LEVELS["compile_run_metrics"] = "INFO"
logger.remove()
logger.add(sys.stderr, filter=_module_filter)
df = compile_all_metrics(dataset_name, stages, params_to_gather, extra_runs=extra_runs, use_cache=True)

2026-03-25 22:20:32.490 | INFO     | scripts.compile_run_metrics:compile_all_metrics:461 - Compiling stage4_sweep_dicece_wts...
2026-03-25 22:20:32.548 | INFO     | scripts.compile_run_metrics:compile_stage_metrics:379 - Compiled [stage4_sweep_dicece_wts/run7, stage4_sweep_dicece_wts/run8, stage4_sweep_dicece_wts/run9]
2026-03-25 22:20:32.550 | INFO     | scripts.compile_run_metrics:compile_all_metrics:461 - Compiling stage5_sweep_dicecewt_nbatch...
2026-03-25 22:20:32.613 | INFO     | scripts.compile_run_metrics:compile_stage_metrics:379 - Compiled [stage5_sweep_dicecewt_nbatch/run1, stage5_sweep_dicecewt_nbatch/run2, stage5_sweep_dicecewt_nbatch/run3, stage5_sweep_dicecewt_nbatch/run4, stage5_sweep_dicecewt_nbatch/run5, stage5_sweep_dicecewt_nbatch/run6]
2026-03-25 22:20:32.614 | INFO     | scripts.compile_run_metrics:compile_all_metrics:461 - Compiling stage3_numcrops_bkd_constwt115...
2026-03-25 22:20:32.665 | INFO     | scripts.compile_run_metrics:compile_stage_metrics:379 - Compi

In [ ]:
compile_all_metrics(dataset_name, stages, params_to_gather, extra_runs=extra_runs, use_cache=False)

In [39]:
reload_recursive(scripts.compile_run_metrics)
from scripts.compile_run_metrics import compile_standalone_run

compile_standalone_run(
    "roi_train1", 
    "roi_train1",
    "/media/smbshare/srs-9/prl_project/training/roi_train1_segresnet",
    params_to_gather,
    True
)

,stage,run,status,learning_rate,crop_ratios,num_crops_per_image,batch_size,weight,include_background,lesion/acc_mean,...,fold4-rim/acc_final,fold4-rim/acc_max,train/loss_mean,train/loss_std,train/loss_min,train/loss_max,val/acc_mean,val/acc_std,val/acc_min,val/acc_max
0,roi_train1,roi_train1_segresnet,complete,0.0002,none,1,1,none,true,0.5975,...,0.4343,0.4662,0.6411,0.066,0.5697,1.0885,0.4603,0.0743,0.1574,0.567


---

In [7]:
exp.dataset

In [16]:
def compile_experiment_metrics(
    experiment: Experiment | Path | str,
    params_to_gather: list[str] | None = None,
    use_cache: bool = True,
) -> dict:
    """ """
    print(type(experiment))
    print(isinstance(experiment, Experiment))
    
    if not isinstance(experiment, Experiment):
        experiment = Experiment.from_run_dir(experiment)

exp = Experiment.from_run_dir(dataset.work_home / "test_dicece_lambda/run1")
compile_experiment_metrics(exp, params_to_gather)

<class 'core.experiment.Experiment'>
True


In [19]:
isinstance(exp, Experiment)

True

In [51]:
thoo = "thee/thoo/thaa"
thoo.replace("/", "_")

'thee_thoo_thaa'

In [40]:
run_params_list = grid.run_params(*grid.get_info())
runs = grid.runs
# run_params_list
runs[0]

{'run_name': 'run1',
 'preprocessing': {},
 'training': {'auto_scale_batch': False,
  'batch_size': 4,
  'loss': {'_target_': 'DiceCELoss',
   'include_background': True,
   'weight': '$torch.tensor([1.0, 1.0, 5.0]).cuda()',
   'lambda_dice': 0.5,
   'lambda_ce': 1.0,
   'squared_pred': True,
   'smooth_nr': 0,
   'smooth_dr': 1e-05,
   'softmax': '$not @sigmoid',
   'sigmoid': '$@sigmoid',
   'to_onehot_y': '$not @sigmoid'}}}

In [41]:
exp1 = Experiment.from_run_dir(f"{grid.experiment_name}/run1", dataset)
exp2 = Experiment.from_run_dir(f"run1", dataset)

In [48]:
exp1.run_dir

PosixPath('/media/smbshare/srs-9/prl_project/training/roi_train2/test_dicece_lambda/run1')

In [45]:
grid.experiments[0].id.partition("/")

('test_dicece_lambda', '/', 'run1')

In [47]:
exp2.id.partition("/")

('run1', '', '')

In [ ]:
"roi_train2#stage4_sweep_dicece_wts"
"roi_train2#run1"
"roi_train1#roi_train1_segresnet"

In [15]:
dataset_name = "roi_train2"
stage = "stage4_sweep_dicece_wts"

dataset = Dataset(dataset_name)
experiment_home = dataset.work_home / stage
grid = ExperimentGrid.from_home_dir(experiment_home)

In [16]:
grid

ExperimentGrid(dataset=roi_train2, experiment=stage4_sweep_dicece_wts, work_home=/media/smbshare/srs-9/prl_project/training/roi_train2/stage4_sweep_dicece_wts)

---

In [79]:
df.sort_values(by="rim/acc_mean", ascending=False)

,batch_size,crop_ratios,include_background,learning_rate,num_crops_per_image,run,stage,status,weight,fold0-rim/acc_final,...,fold4-lesion/acc_max,lesion/acc_max,lesion/acc_mean,lesion/acc_std,train/loss_mean,train/loss_min,train/loss_std,val/acc_max,val/acc_mean,val/acc_std
10,1,none,false,0.0002,1,stage3_numcrops_bkd_constwt115/run2,stage3_numcrops_bkd_constwt115,complete,"1.0,1.0,5.0",0.4271,...,0.6711,0.6987,0.6437,0.0411,3.8195,3.7237,0.0899,0.6580,0.5390,0.0610
3,2,none,false,0.0002,4,stage5_sweep_dicecewt_nbatch/run1,stage5_sweep_dicecewt_nbatch,complete,"1.0,1.0,5.0",0.3880,...,0.6614,0.6982,0.6363,0.0457,3.8286,3.7311,0.1078,0.6552,0.5307,0.0683
12,1,none,false,0.0002,4,stage3_numcrops_bkd_constwt115/run4,stage3_numcrops_bkd_constwt115,complete,"1.0,1.0,5.0",0.3891,...,0.6713,0.7063,0.6398,0.0490,3.8283,3.7315,0.1046,0.6605,0.5311,0.0693
25,1,none,false,0.0002,2,stage2_numcrops_dicece/run4,stage2_numcrops_dicece,complete,"1.0,1.0,5.0",0.3708,...,0.6583,0.7083,0.6373,0.0451,3.8278,3.7322,0.0980,0.6675,0.5298,0.0671
9,1,none,true,0.0002,1,stage3_numcrops_bkd_constwt115/run1,stage3_numcrops_bkd_constwt115,complete,"1.0,1.0,5.0",0.3579,...,0.6588,0.7065,0.6456,0.0417,2.5542,2.4865,0.0637,0.6649,0.5334,0.0629
24,1,none,true,0.0002,2,stage2_numcrops_dicece/run3,stage2_numcrops_dicece,complete,"1.0,1.0,5.0",0.3612,...,0.6557,0.6986,0.6372,0.0428,2.5615,2.4939,0.0706,0.6582,0.5266,0.0652
6,2,none,true,0.0002,4,stage5_sweep_dicecewt_nbatch/run4,stage5_sweep_dicecewt_nbatch,complete,"1.0,1.0,5.0",0.3868,...,0.6582,0.6976,0.6370,0.0473,2.5614,2.4932,0.0785,0.6478,0.5252,0.0674
11,1,none,true,0.0002,4,stage3_numcrops_bkd_constwt115/run3,stage3_numcrops_bkd_constwt115,complete,"1.0,1.0,5.0",0.4027,...,0.6663,0.7017,0.6400,0.0481,2.5615,2.4909,0.0761,0.6551,0.5264,0.0693
26,1,none,true,0.0002,4,test_dicece_lambda/run1,test_dicece_lambda,complete,"1.0,1.0,5.0",0.2770,...,0.6707,0.7073,0.6416,0.0503,1.2934,1.2532,0.0460,0.6612,0.5171,0.0739
0,1,none,true,0.0002,1,stage4_sweep_dicece_wts/run7,stage4_sweep_dicece_wts,complete,"1.0,2.0,5.0",0.3523,...,0.6682,0.7147,0.6459,0.0484,2.6781,2.5621,0.1286,0.6527,0.5190,0.0733


In [ ]:
dataset_name = "roi_train2"
for gridsearch_name in gridsearch_experiments:
    # gridsearch_name = "stage5_sweep_dicecewt_nbatch"
    grid = load_grid(dataset_name, gridsearch_name)
    run_params = grid.run_params(*grid.get_info())
    ds = Dataset(dataset_name)
    runs_to_check = ["run1"]
    runs = []
    for run_info in run_params:
        run_loc = f"{grid.experiment_name}/{run_info['run_name']}"
        print(f"STARTING {run_loc}")
        exp = Experiment.from_run_dir(run_loc, ds)
        exp.predict()
        exp.cases #builds the case list for the first time and caches for quick access
        runs.append({**run_info, "exp": exp})

# clear_output()

In [ ]:
# should return a str like "$torch.tensor([1.0, 3.0, 5.0]).cuda()"
wt_str = runs[0]["exp"].hyper_params["loss#weight"] 
# The tensor can be made to be nicer for analysis tables with
str_tensor = re.match(r"\$torch\.tensor\(\[(.+)\]\).+", wt_str)[1]
weights = [float(w) for w in str_tensor.split(",")]


# should return a boolean
runs[0]["exp"].hyper_params["loss#include_background"] 

# this is an old function I wrote, im not using it anymore, but its what
def get_loss_param(self, param):
    import re
    val = self.loss.get(param, None)
    if isinstance(val, str) and "torch" in val:
        str_tensor = re.match(r"\$torch\.tensor\(\[(.+)\]\).+", val)[1]
        return [float(w) for w in str_tensor.split(",")]
    else:
        return val

1

In [ ]:
gridsearch_name = "stage5_sweep_dicecewt_nbatch"
# I had written the function get_info() in grid.py to get the grid parameters
# in a more table friendly format, but it depended on the user specifying table friendly
#  keys within the experiment.yaml definition. Instead, lets just define a mapping for each key and type of value
#   in one spot as a function or class. It can be a growing function as I try to test more hyperparameters in the future
run_params = grid.run_params(*grid.get_info())

ds = Dataset(dataset_name)
runs_to_check = ["run1"]
runs = []
for run_info in run_params:
    run_loc = f"{grid.experiment_name}/{run_info['run_name']}"
    print(f"STARTING {run_loc}")
    exp = Experiment.from_run_dir(run_loc, ds)
    exp.predict()
    exp.cases #builds the case list for the first time and caches for quick access
    runs.append({**run_info, "exp": exp})


def compile_metrics(runs):
    compiled_metrics = defaultdict(list)
    for run in tqdm(runs, total=len(runs)):
        compiled_metrics["run"].append(run["run_name"])
        for hyperparam_key, hyperparam_val in run["training"].items():
            compiled_metrics[hyperparam_key].append(hyperparam_val)
        fold_data = analyze_unified_mlruns(run["exp"].run_dir)
        aggregated = aggregate_metrics(fold_data)
        by_prefix = defaultdict(dict)
        for metric_name in aggregated:
            if "/" in metric_name:
                prefix = metric_name.split("/")[0]
                by_prefix[prefix][metric_name] = aggregated[metric_name]
            else:
                by_prefix["other"][metric_name] = aggregated[metric_name]

        for prefix in sorted(by_prefix.keys()):
            for metric_name in sorted(by_prefix[prefix].keys()):
                metric_data = aggregated[metric_name]

                if "stats" not in metric_data:
                    continue

                stats = metric_data["stats"]
                for stat in ["mean", "std", "min", "max"]:
                    compiled_metrics[f"{metric_name}_{stat}"].append(
                        np.round(stats[stat], 4)
                    )

                fold_values = [
                    (fold_num, metric_data[fold_num])
                    for fold_num in sorted(fold_data.keys())
                    if fold_num in metric_data
                ]
                if "val_class" not in metric_name:
                    continue
                for fold_num, values in fold_values:
                    compiled_metrics[f"fold{fold_num}-{metric_name}_final"].append(
                        np.round(values[-1], 4)
                    )
                    compiled_metrics[f"fold{fold_num}-{metric_name}_max"].append(
                        np.round(np.max(values), 4)
                    )

    metrics = pd.DataFrame(compiled_metrics).set_index("run")
    col_remap = {}
    for col in metrics.columns:
        # lesion accs
        new_col = re.sub(r"^(.*)(val_)(class/acc_0)(_.+)", r"\1lesion/acc\4", col)
        col_remap[col] = new_col
        # rim accs
        col_remap[col] = re.sub(
            r"^(.*)(val_)(class/acc_1)(_.+)", r"\1rim/acc\4", new_col
        )

    metrics = metrics.rename(columns=col_remap)
    col_order_cats = defaultdict(list)
    for col in metrics.columns:
        if "acc_min" in col or "loss_max" in col:
            continue
        if "rim" in col:
            col_order_cats["rim"].append(col)
        elif "lesion" in col:
            col_order_cats["lesion"].append(col)
        elif "train" in col:
            col_order_cats["loss"].append(col)
        elif "val" in col:
            col_order_cats["val"].append(col)
        else:
            col_order_cats["run_info"].append(col)

    column_order = []
    for cat in ["run_info", "rim", "lesion", "loss", "val"]:
        column_order.extend(col_order_cats[cat])
    metrics = metrics[column_order]

    return metrics


  0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
metrics = compile_metrics(runs_to_analyze)
metrics.to_csv(f"{gridsearch_name}_metrics.csv")

In [ ]:
import pandas as pd

metrics = pd.DataFrame(compiled_metrics).set_index("run")
col_remap = {}
for col in metrics.columns:
    # lesion accs
    new_col = re.sub(r"^(.*)(val_)(class/acc_0)(_.+)", r"\1lesion/acc\4", col)
    col_remap[col] = new_col
    # rim accs
    col_remap[col] = re.sub(r"^(.*)(val_)(class/acc_1)(_.+)", r"\1rim/acc\4", new_col)

metrics = metrics.rename(columns=col_remap)
col_order_cats = defaultdict(list)
for col in metrics.columns:
    if "acc_min" in col or "loss_max" in col:
        continue
    if "rim" in col:
        col_order_cats["rim"].append(col)
    elif "lesion" in col:
        col_order_cats["lesion"].append(col)
    elif "train" in col:
        col_order_cats["loss"].append(col)
    elif "val" in col:
        col_order_cats["val"].append(col)
    else:
        col_order_cats["run_info"].append(col)

column_order = []
for cat in ["run_info", "rim", "lesion", "loss", "val"]:
    column_order.extend(col_order_cats[cat])
metrics = metrics[column_order]
metrics.to_csv(f"{gridsearch_name}_metrics.csv")

In [ ]:
metrics2 = metrics.copy()
metrics2.insert(2, "rim_dice_mean (PRL test cases)", None)
metrics2.insert(3, "rim_dice_std (PRL test cases)", None)
metrics2.insert(4, "lesion_dice_mean (PRL test cases)", None)
metrics2.insert(5, "lesion_dice_std (PRL test cases)", None)

for run in tqdm(runs, total=len(runs)):
    exp = run["exp"]
    testing_cases = [
        item
        for item in exp.cases
        if item["split"] == "testing" and item["case_type"] == "PRL"
    ]
    rim_dice = []
    lesion_dice = []
    for item in testing_cases:
        rim_dice.append(
            utils.dice_score(item["label"], item["inference"], seg1_val=2, seg2_val=2)
        )
        lesion_dice.append(
            utils.dice_score(item["label"], item["inference"], seg1_val=1, seg2_val=1)
        )
    metrics2.loc[run["run_name"], "rim_dice_mean (PRL test cases)"] = np.mean(rim_dice)
    metrics2.loc[run["run_name"], "rim_dice_std (PRL test cases)"] = np.std(rim_dice)
    metrics2.loc[run["run_name"], "lesion_dice_mean (PRL test cases)"] = np.mean(
        lesion_dice
    )
    metrics2.loc[run["run_name"], "lesion_dice_std (PRL test cases)"] = np.std(
        lesion_dice
    )
metrics.to_csv(f"{gridsearch_name}_metrics_w_dice.csv")

  0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
compiled_metrics = defaultdict(list)
for run in runs:
    compiled_metrics["run"].append(run["run_name"])
    compiled_metrics["lr"].append(exp.training_config.learning_rate)
    crop_ratios = exp.training_config.crop_ratios
    if crop_ratios is not None:
        compiled_metrics["crop_ratios"].append(",".join([str(x) for x in crop_ratios]))
    else:
        compiled_metrics["crop_ratios"].append("null")
    fold_data = analyze_unified_mlruns(exp.run_dir)
    aggregated = aggregate_metrics(fold_data)
    by_prefix = defaultdict(dict)
    for metric_name in aggregated:
        if "/" in metric_name:
            prefix = metric_name.split("/")[0]
            by_prefix[prefix][metric_name] = aggregated[metric_name]
        else:
            by_prefix["other"][metric_name] = aggregated[metric_name]

    for prefix in sorted(by_prefix.keys()):
        for metric_name in sorted(by_prefix[prefix].keys()):
            metric_data = aggregated[metric_name]

            if "stats" not in metric_data:
                continue

            stats = metric_data["stats"]
            for stat in ["mean", "std", "min", "max"]:
                compiled_metrics[f"{metric_name}_{stat}"].append(stats[stat])

            fold_values = [
                (fold_num, metric_data[fold_num])
                for fold_num in sorted(fold_data.keys())
                if fold_num in metric_data
            ]
            if "val_class" not in metric_name:
                continue
            for fold_num, values in fold_values:
                compiled_metrics[f"fold{fold_num}-{metric_name}_final"].append(
                    values[-1]
                )
                compiled_metrics[f"fold{fold_num}-{metric_name}_max"].append(
                    np.max(values)
                )

In [54]:
metrics2.to_csv("gridsearch_metrics_w_dice.csv")

In [ ]:
run_dir = runs[1].run_dir
datastats = fold_dx.load_datastats(run_dir)
rows = []
for item in runs[1].cases:
    if item["split"] == "testing":
        continue
    rows.append({k: item[k] for k in ["subid", "lesion_index", "split", "case_type"]})
metadata_df = pd.DataFrame(rows)
df = metadata_df.merge(datastats, on=["subid", "lesion_index"], how="left")

Loading /media/smbshare/srs-9/prl_project/training/roi_train2/stage1_crop_lr_sweep/run1/datastats_by_case.yaml ...
  Loaded 794 cases from datastats


In [ ]:
all_numerical_cols = df.select_dtypes(include="number").columns
ignore_numerical_cols = ["subid", "lesion_index"]
cols_to_mean = all_numerical_cols[~all_numerical_cols.isin(ignore_numerical_cols)]
means_by_fold = df.groupby("split")[cols_to_mean].mean()

In [55]:
means_by_fold.to_csv("fold_image_stats.csv")

In [ ]:
col_remap = {
    "val_class/acc_0_mean": "val_lesion/acc_mean",
    "val_class/acc_0_std": "val_lesion/acc_std",
    "val_class/acc_0_min": "val_lesion/acc_min",
    "val_class/acc_0_max": "val_lesion/acc_max",
    "val_class/acc_1_mean": "val_rim/acc_mean",
    "val_class/acc_1_std": "val_rim/acc_std",
    "val_class/acc_1_min": "val_rim/acc_min",
    "val_class/acc_1_max": "val_rim/acc_max",
}
col_order = [
    "lr",
    "crop_ratios",
    "val_rim/acc_mean",
    "val_rim/acc_max",
    "val_rim/acc_std",
    "val_lesion/acc_mean",
    "val_lesion/acc_max",
    "val_lesion/acc_std",
    "train/loss_mean",
    "train/loss_min",
    "train/loss_std",
    "val/acc_mean",
    "val/acc_max",
    "val/acc_std",
]
metrics = metrics.rename(columns=col_remap)[col_order]


In [62]:
round_columns = metrics.columns[~metrics.columns.isin(["lr", "run", "crop_ratios"])]
metrics[round_columns] = metrics[round_columns].round(3)

In [ ]:
col_remap = {
    "train/loss_mean",
    "train/loss_std",
    "train/loss_min",
    "train/loss_max",
    "val/acc_mean",
    "val/acc_std",
    "val/acc_min",
    "val/acc_max",
    "val_lesion/acc_mean",
    "val_lesion/acc_std",
    "val_lesion/acc_min",
    "val_lesion/acc_max",
    "val_rim/acc_mean",
    "val_rim/acc_std",
    "val_rim/acc_min",
    "val_rim/acc_max",
}

Index(['lr', 'crop_ratios', 'train/loss_mean', 'train/loss_std',
       'train/loss_min', 'train/loss_max', 'val/acc_mean', 'val/acc_std',
       'val/acc_min', 'val/acc_max', 'val_class/acc_0_mean',
       'val_class/acc_0_std', 'val_class/acc_0_min', 'val_class/acc_0_max',
       'val_class/acc_1_mean', 'val_class/acc_1_std', 'val_class/acc_1_min',
       'val_class/acc_1_max'],
      dtype='object')

In [40]:
performance_metrics = runs[1].evaluate()

/media/smbshare/srs-9/prl_project/data/sub2131-20190117/26/lesion_xy20_z2.nii.gz /media/smbshare/srs-9/prl_project/training/roi_train2/stage1_crop_lr_sweep/run1/ensemble_output/sub2131-20190117/26/flair.phase_xy20_z2_ensemble.nii.gz
/media/smbshare/srs-9/prl_project/data/sub1010-20180208/27/lesion_xy20_z2.nii.gz /media/smbshare/srs-9/prl_project/training/roi_train2/stage1_crop_lr_sweep/run1/ensemble_output/sub1010-20180208/27/flair.phase_xy20_z2_ensemble.nii.gz
/media/smbshare/srs-9/prl_project/data/sub1293-20161129/70/lesion_xy20_z2.nii.gz /media/smbshare/srs-9/prl_project/training/roi_train2/stage1_crop_lr_sweep/run1/ensemble_output/sub1293-20161129/70/flair.phase_xy20_z2_ensemble.nii.gz
194 valid cases of 197
/media/smbshare/srs-9/prl_project/data/sub1050-20170624/86/lesion_xy20_z2.nii.gz /media/smbshare/srs-9/prl_project/training/roi_train2/stage1_crop_lr_sweep/run1/fold_predictions/fold0/sub1050-20170624/86/flair.phase_xy20_z2.nii.gz
/media/smbshare/srs-9/prl_project/data/sub1050-

In [42]:
runs[1].cases[0]

{'subid': 1293,
 'lesion_index': 1,
 'image': PosixPath('/media/smbshare/srs-9/prl_project/data/sub1293-20161129/1/flair.phase_xy20_z2.nii.gz'),
 'label': PosixPath('/media/smbshare/srs-9/prl_project/data/sub1293-20161129/1/prl_label_SRS_CH_xy20_z2.nii.gz'),
 'split': 'testing',
 'inference': PosixPath('/media/smbshare/srs-9/prl_project/training/roi_train2/stage1_crop_lr_sweep/run1/ensemble_output/sub1293-20161129/1/flair.phase_xy20_z2_ensemble.nii.gz')}

In [41]:
performance_metrics

,subid,lesion_index,split,case_type,lesion_dice,prl_dice,tp,fp,tn,fn,sensitivity,specificity,fpr,fnr,precision,npv,accuracy,f1,Notes
0,1293,1,testing,None,0.888445,0.328682,106,152,7347,182,0.368056,0.979731,0.020269,0.631944,0.410853,0.975827,0.957108,0.388278,
1,1074,3,testing,None,0.678937,0.509165,125,76,166,34,0.786164,0.685950,0.314050,0.213836,0.621891,0.830000,0.725686,0.694444,
2,1080,3,testing,None,0.706790,0.528302,140,116,229,20,0.875000,0.663768,0.336232,0.125000,0.546875,0.919679,0.730693,0.673077,
3,2131,1,testing,None,0.866130,0.695710,519,198,1184,62,0.893287,0.856729,0.143271,0.106713,0.723849,0.950241,0.867550,0.799692,
4,1011,7,testing,None,0.765677,0.570922,161,100,232,9,0.947059,0.698795,0.301205,0.052941,0.616858,0.962656,0.782869,0.747100,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
986,1078,12,validation fold4,None,0.761905,0.000000,0,1,168,0,NaN,0.994083,0.005917,NaN,0.000000,1.000000,0.994083,NaN,
987,1050,82,validation fold4,None,0.285714,NaN,0,0,1,0,NaN,1.000000,0.000000,NaN,NaN,1.000000,1.000000,NaN,
988,1293,67,validation fold4,None,0.615385,NaN,0,0,4,0,NaN,1.000000,0.000000,NaN,NaN,1.000000,1.000000,NaN,
989,1125,96,validation fold4,None,0.602410,NaN,0,0,150,0,NaN,1.000000,0.000000,NaN,NaN,1.000000,1.000000,NaN,
